# Train YOLOv8n for cenital head detection (Phase A)

Phase A of the People Counter retraining: fine-tune `yolov8n` on the Roboflow `overhead_person` (or equivalent) dataset, then export to ONNX so it can be compiled to a Hailo-8L `.hef` on a workstation (WSL2 + Hailo Dataflow Compiler).

**Runtime:** Colab → `Runtime` → `Change runtime type` → `T4 GPU` (free tier).

**Required Colab Secrets** (left sidebar 🔑 icon):
- `ROBOFLOW_API_KEY`

**Manual edits before running:**
1. Set `WORKSPACE`, `PROJECT`, `VERSION` in cell 4 (matches the URL of your Roboflow Universe dataset).
2. (optional) Adjust `EPOCHS`, `BATCH`, `IMGSZ` in cell 5.

Total wall time on T4: ~2-4 h for ~5k images × 50 epochs. Checkpoints save to Drive every 10 epochs so a disconnect is recoverable.

## 1. Mount Google Drive (for checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/people-counter-training'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive workspace:', DRIVE_ROOT)

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics==8.3.* roboflow==1.1.*
import ultralytics; ultralytics.checks()

## 3. Read the Roboflow API key from Colab Secrets

Add `ROBOFLOW_API_KEY` to Colab Secrets (🔑 icon, left sidebar) so it isn't baked into the notebook.

In [ ]:
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, 'Add ROBOFLOW_API_KEY to Colab Secrets and re-run this cell.'
print('API key loaded (length =', len(ROBOFLOW_API_KEY), ')')

## 4. Download the Roboflow dataset

Edit the three slugs below to match your dataset's URL:
`https://universe.roboflow.com/<WORKSPACE>/<PROJECT>/<VERSION>`

In [ ]:
WORKSPACE = 'REPLACE-ME-workspace-slug'
PROJECT   = 'REPLACE-ME-project-slug'
VERSION   = 1

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download('yolov8',
    location=f'/content/dataset/{WORKSPACE}__{PROJECT}__v{VERSION}')
DATA_YAML = f'{dataset.location}/data.yaml'
print('Dataset at:', dataset.location)
print('data.yaml :', DATA_YAML)
!cat "$DATA_YAML"

## 5. Train

We start from `yolov8n.pt` (COCO pretrained — has a `person` class which is a useful prior even though we'll re-target to head/overhead). Owen718's CrowdHuman head weights would be a stronger starting point but require a manual download from their GitHub release; if you want to swap, replace the `model=` argument with the path to those weights.

In [ ]:
EPOCHS = 50
BATCH  = 32
IMGSZ  = 640
PROJECT_DIR = f'{DRIVE_ROOT}/runs'
RUN_NAME    = f'{PROJECT}_v{VERSION}_yolov8n'

from ultralytics import YOLO
model = YOLO('yolov8n.pt')   # COCO-pretrained starter

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    save_period=10,        # checkpoint every 10 epochs (Drive-resilient)
    patience=20,           # early-stop if val mAP stops improving
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    seed=42,
    verbose=True,
)
BEST_PT = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
print('Best weights at:', BEST_PT)

## 6. Validate

In [ ]:
from ultralytics import YOLO
best = YOLO(BEST_PT)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, plots=True)
print('mAP50    :', float(metrics.box.map50))
print('mAP50-95 :', float(metrics.box.map))
print('Precision:', float(metrics.box.mp))
print('Recall   :', float(metrics.box.mr))

## 7. Export to ONNX (for Hailo compilation)

Hailo Dataflow Compiler (`hailomz compile`) takes ONNX as input. We export with `opset=12` (Hailo-friendly) and dynamic batch off (HEF needs a fixed graph).

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, dynamic=False, simplify=True)
ONNX_PATH = BEST_PT.replace('.pt', '.onnx')
print('ONNX:', ONNX_PATH)
import os; print('size (MB):', round(os.path.getsize(ONNX_PATH) / (1024*1024), 2))

## 8. Stash ONNX + a calibration set on Drive

We also copy ~200 random training images to Drive — the Hailo compiler uses them as a quantization calibration set when compiling to int8 HEF.

In [ ]:
import shutil, random
EXPORT_DIR = f'{DRIVE_ROOT}/export/{RUN_NAME}'
os.makedirs(EXPORT_DIR, exist_ok=True)

shutil.copy(ONNX_PATH, f'{EXPORT_DIR}/best.onnx')
shutil.copy(BEST_PT,   f'{EXPORT_DIR}/best.pt')
shutil.copy(DATA_YAML, f'{EXPORT_DIR}/data.yaml')

# Calibration set
calib_dir = f'{EXPORT_DIR}/calib'
os.makedirs(calib_dir, exist_ok=True)
import glob
train_imgs = glob.glob(f'{dataset.location}/train/images/*')
random.seed(42)
for src in random.sample(train_imgs, min(200, len(train_imgs))):
    shutil.copy(src, calib_dir)

print('Stashed to Drive at:', EXPORT_DIR)
!ls -la "$EXPORT_DIR"

## Next step (off-Colab): compile to Hailo HEF

Download `EXPORT_DIR` from Drive to your WSL2 Ubuntu workstation, then:

```bash
# Inside WSL2 with hailo-dataflow-compiler installed
hailomz compile yolov8n \
    --ckpt best.onnx \
    --hw-arch hailo8l \
    --calib-path calib/
```

Output: `yolov8n.hef`. SCP that to the Pi and update `detection.model_path` in `/etc/people-counter/config.yaml`.